In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *
import sys
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df_sales = spark.read.format("CSV")\
    .option('header', 'True')\
    .option('inferSchema', 'True')\
    .load(r'/Volumes/ska_catalog/bronze/bronze_volume/raw_data/sales/sales_data.csv')

df_sales.display()

In [0]:
df_sales.columns

In [0]:
def clean_col_nme (df : object, col_nme : list) -> object:
    for col in col_nme:
        df = df.withColumnRenamed(col, col.strip().lower().replace(' ', ''))
    return df

In [0]:
df_sales = clean_col_nme(df_sales, df_sales.columns)
df_sales.display()

In [0]:
for col_nme in df_sales.columns:
    df_sales = df_sales.withColumn(col_nme, regexp_replace(col(col_nme), "\\$", ""))
    df_sales = df_sales.withColumn(col_nme, regexp_replace(col(col_nme), ",", ""))

df_sales = df_sales.withColumn('unitssold', col('unitssold').cast(IntegerType()))\
    .withColumn('manufacturingprice', col('manufacturingprice').cast(FloatType()))\
    .withColumn('saleprice', col('saleprice').cast(FloatType()))\
    .withColumn('grosssales', col('grosssales').cast(FloatType()))\
    .withColumn('discounts', col('discounts').cast(FloatType()))\
    .withColumn('sales', col('sales').cast(FloatType()))\
    .withColumn('cogs',col('cogs').cast(FloatType()))\
    .withColumn('profit',col('profit').cast(FloatType()))\
    .withColumn('date', to_timestamp(col("date"), "M/d/yyyy"))
df_sales.display()

In [0]:
from pyspark.sql.functions import round, month, monthname, year

int_cols = [field.name for field in df_sales.schema.fields if isinstance(field.dataType, IntegerType)]
for col_nme in int_cols:
    df_sales = df_sales.withColumn(col_nme, round(col(col_nme),2))


df_sales = df_sales.withColumn("monthnumber", month(col("date")))\
    .withColumn("monthname", date_format(col("date"), "MMMM"))\
    .withColumn("year", year(col("date")))

df_sales.display()

In [0]:
len(df_sales.columns)